# Week 08 · Thursday — RNNs + Sequential Data
**PG Diploma · AI-ML & Agentic AI Engineering · IIT Gandhinagar**

**Topics:** RNNs, LSTMs, GRU, BiRNN — Architecture, Vanishing Gradients, Sequence Prediction  
**Dataset:** `stock_prices.csv` (5 Indian equities, Jan 2022 – Nov 2024)

---
> **Sub-steps covered:** 1 (Easy), 3 (Medium), 5 (Medium), 6 (Hard — optional)  
> Sub-steps 2 & 4 require `chat_logs.csv` which was not available — placeholders included.

## 0. Imports & Configuration

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ── Constants (no magic numbers) ──────────────────────────────────────────────
STOCK_TICKER      = "RELIANCE"   # stock chosen for Sub-steps 1, 3, 6
WINDOW_SIZE       = 30           # lookback window in trading days (~6 weeks)
TRAIN_RATIO       = 0.80         # chronological split: 80% train
VAL_RATIO         = 0.10         # 10% validation, 10% test
BATCH_SIZE        = 32
LEARNING_RATE     = 1e-3
NUM_EPOCHS        = 50
LSTM_HIDDEN_SIZE  = 64
LSTM_NUM_LAYERS   = 2
LSTM_DROPOUT      = 0.2
RANDOM_SEED       = 42
DATA_PATH         = "stock_prices.csv"

# Reproducibility
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

---
## Sub-step 1 🟢 — Stock Data: Sequence Construction & Split Strategy

### Design Decisions (document before code)

**Window size — 30 trading days**  
A 30-day (~6-week) lookback captures meaningful medium-term momentum and mean-reversion patterns without introducing excessive noise from very distant history. Shorter windows (e.g., 5 days) miss weekly/monthly seasonality; longer windows (e.g., 90 days) add noise and slow training.

**Split strategy — strictly chronological (never random)**  
Time-series data must be split chronologically — train on the past, validate/test on the future. Using a random split would leak future information into training features (look-ahead bias), inflating reported performance. In finance, this is a career-ending error because the deployed model would never see that future data in production.  
→ 80% train | 10% validation | 10% test — all in time order.

**What a random split would do**  
Random splitting creates data leakage: sequences in the test set would contain windows partially drawn from the future relative to the training set. The model would appear highly accurate (e.g., R² > 0.9) because it effectively "memorises" the trend direction from leaked future data — but would perform near-randomly in live trading.

In [ ]:
def load_and_filter_stock(path: str, ticker: str) -> pd.DataFrame:
    """Load CSV and return single-ticker DataFrame sorted by date."""
    try:
        df = pd.read_csv(path, parse_dates=["date"])
    except FileNotFoundError as e:
        raise FileNotFoundError(f"Dataset not found at '{path}'. Check DATA_PATH.") from e

    df_ticker = df[df["ticker"] == ticker].copy()
    if df_ticker.empty:
        raise ValueError(f"Ticker '{ticker}' not found. Available: {df['ticker'].unique()}")

    df_ticker.sort_values("date", inplace=True)
    df_ticker.reset_index(drop=True, inplace=True)
    return df_ticker


def chronological_split(df: pd.DataFrame, train_ratio: float, val_ratio: float):
    """Split DataFrame chronologically into train / val / test."""
    n = len(df)
    train_end = int(n * train_ratio)
    val_end   = int(n * (train_ratio + val_ratio))
    return df.iloc[:train_end], df.iloc[train_end:val_end], df.iloc[val_end:]


def build_sequences(prices: np.ndarray, window: int):
    """Create (X, y) sliding-window sequences from a 1-D price array."""
    X, y = [], []
    for i in range(len(prices) - window):
        X.append(prices[i : i + window])
        y.append(prices[i + window])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


# ── Load & inspect ─────────────────────────────────────────────────────────────
df_stock = load_and_filter_stock(DATA_PATH, STOCK_TICKER)
print(f"Loaded {len(df_stock)} rows for {STOCK_TICKER}")
print(f"Date range: {df_stock['date'].min().date()} → {df_stock['date'].max().date()}")
print(df_stock[["date","open","high","low","close","volume","returns_pct"]].head())

In [ ]:
# ── Scale close prices ─────────────────────────────────────────────────────────
# Scaler fitted ONLY on training data to avoid leakage
train_df, val_df, test_df = chronological_split(df_stock, TRAIN_RATIO, VAL_RATIO)

scaler = MinMaxScaler(feature_range=(0, 1))
train_prices = scaler.fit_transform(train_df[["close"]]).flatten()
val_prices   = scaler.transform(val_df[["close"]]).flatten()
test_prices  = scaler.transform(test_df[["close"]]).flatten()

print(f"Train: {len(train_df)} rows ({train_df['date'].min().date()} – {train_df['date'].max().date()})")
print(f"Val  : {len(val_df)}  rows ({val_df['date'].min().date()} – {val_df['date'].max().date()})")
print(f"Test : {len(test_df)} rows ({test_df['date'].min().date()} – {test_df['date'].max().date()})")

# ── Build sequences ────────────────────────────────────────────────────────────
X_train, y_train = build_sequences(train_prices, WINDOW_SIZE)
X_val,   y_val   = build_sequences(val_prices,   WINDOW_SIZE)
X_test,  y_test  = build_sequences(test_prices,  WINDOW_SIZE)

print(f"\nSequence shapes — X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"                  X_test:  {X_test.shape},  y_test:  {y_test.shape}")

In [ ]:
# ── Visualise the split ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(train_df["date"], train_df["close"], label="Train",      color="steelblue",  linewidth=1.2)
ax.plot(val_df["date"],   val_df["close"],   label="Validation", color="darkorange", linewidth=1.2)
ax.plot(test_df["date"],  test_df["close"],  label="Test",       color="crimson",    linewidth=1.2)
ax.set_title(f"{STOCK_TICKER} — Chronological Train / Val / Test Split", fontsize=13)
ax.set_xlabel("Date"); ax.set_ylabel("Close Price (₹)")
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.tight_layout()
plt.savefig("split_visualisation.png", dpi=120)
plt.show()

---
## Sub-step 2 🟢 — Chat Logs EDA *(placeholder — chat_logs.csv not provided)*

> `chat_logs.csv` was not available in the provided files. The cell below shows the intended approach; it will run once the file is placed in the same directory.

**Known issue:** The timestamp column uses mixed formats (e.g., `DD/MM/YYYY HH:MM` and ISO 8601 strings). Fix: use `pd.to_datetime(..., infer_datetime_format=True, dayfirst=True)` with a fallback to `errors='coerce'` followed by a second parse pass for remaining NaTs.

In [ ]:
import os

CHAT_PATH = "chat_logs.csv"

if not os.path.exists(CHAT_PATH):
    print("[INFO] chat_logs.csv not found — skipping Sub-step 2.")
    print("Place chat_logs.csv in the same directory and re-run.")
else:
    def fix_mixed_timestamps(series: pd.Series) -> pd.Series:
        """Parse mixed-format timestamp column robustly."""
        parsed = pd.to_datetime(series, infer_datetime_format=True,
                                dayfirst=True, errors="coerce")
        unparsed_mask = parsed.isna()
        if unparsed_mask.sum() > 0:
            fallback = pd.to_datetime(series[unparsed_mask],
                                      format="%m/%d/%Y %H:%M", errors="coerce")
            parsed[unparsed_mask] = fallback
        remaining_nulls = parsed.isna().sum()
        print(f"  Timestamp parse: {len(series)} rows, {remaining_nulls} still-null after two passes.")
        return parsed

    df_chat = pd.read_csv(CHAT_PATH)
    df_chat["timestamp"] = fix_mixed_timestamps(df_chat["timestamp"])
    print(df_chat.head())
    # EDA: churn signal analysis would follow here
    print("[Sub-step 2] Timestamp fixed. Proceed with churn EDA.")

---
## Sub-step 3 🟡 — LSTM for Next-Day Stock Price Prediction

In [ ]:
class StockSequenceDataset(Dataset):
    """PyTorch Dataset wrapping (X, y) numpy arrays for LSTM input."""
    def __init__(self, X: np.ndarray, y: np.ndarray):
        # LSTM expects (batch, seq_len, features) — add feature dim
        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(-1)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(-1)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class StockLSTM(nn.Module):
    """Stacked LSTM with dropout for next-day close price regression.

    Architecture rationale:
    - 2 LSTM layers: first layer learns low-level price patterns;
      second learns higher-order temporal dependencies.
    - Dropout(0.2) between layers: regularises against overfitting on
      the relatively small ~600-sample training set.
    - Single linear output: regression task, no activation on output.
    """
    def __init__(self, input_size: int = 1, hidden_size: int = LSTM_HIDDEN_SIZE,
                 num_layers: int = LSTM_NUM_LAYERS, dropout: float = LSTM_DROPOUT):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)       # (batch, seq_len, hidden)
        last_step   = lstm_out[:, -1, :] # take final timestep
        return self.fc(last_step)        # (batch, 1)


# ── DataLoaders ────────────────────────────────────────────────────────────────
train_dataset = StockSequenceDataset(X_train, y_train)
val_dataset   = StockSequenceDataset(X_val,   y_val)
test_dataset  = StockSequenceDataset(X_test,  y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")

In [ ]:
def train_epoch(model, loader, criterion, optimiser):
    """Run one training epoch; return mean loss."""
    model.train()
    running_loss = 0.0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        optimiser.zero_grad()
        preds = model(X_batch)
        loss  = criterion(preds, y_batch)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # gradient clipping
        optimiser.step()
        running_loss += loss.item() * len(y_batch)
    return running_loss / len(loader.dataset)


def evaluate(model, loader, criterion):
    """Evaluate model on a DataLoader; return mean loss."""
    model.eval()
    running_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            preds = model(X_batch)
            running_loss += criterion(preds, y_batch).item() * len(y_batch)
    return running_loss / len(loader.dataset)


def predict_all(model, loader):
    """Return numpy arrays of (predictions, actuals) in scaled space."""
    model.eval()
    preds_list, actuals_list = [], []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            preds_list.append(model(X_batch.to(DEVICE)).cpu().numpy())
            actuals_list.append(y_batch.numpy())
    return np.concatenate(preds_list).flatten(), np.concatenate(actuals_list).flatten()


# ── Instantiate model ──────────────────────────────────────────────────────────
lstm_model = StockLSTM().to(DEVICE)
criterion  = nn.MSELoss()
optimiser  = torch.optim.Adam(lstm_model.parameters(), lr=LEARNING_RATE)
scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(optimiser, patience=5, factor=0.5)

print(lstm_model)

In [ ]:
# ── Training loop with early stopping ─────────────────────────────────────────
PATIENCE       = 10   # early-stopping patience
best_val_loss  = float("inf")
patience_count = 0
train_losses, val_losses = [], []

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_epoch(lstm_model, train_loader, criterion, optimiser)
    val_loss   = evaluate(lstm_model, val_loader, criterion)
    scheduler.step(val_loss)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_count = 0
        torch.save(lstm_model.state_dict(), "best_lstm.pt")
    else:
        patience_count += 1
        if patience_count >= PATIENCE:
            print(f"Early stopping at epoch {epoch}.")
            break

    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{NUM_EPOCHS} — Train MSE: {train_loss:.6f} | Val MSE: {val_loss:.6f}")

# Load best checkpoint
lstm_model.load_state_dict(torch.load("best_lstm.pt", map_location=DEVICE))
print(f"\nBest val MSE: {best_val_loss:.6f}")

In [ ]:
# ── Training curve ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train_losses, label="Train MSE", color="steelblue")
ax.plot(val_losses,   label="Val MSE",   color="darkorange")
ax.set_title("LSTM Training Curve")
ax.set_xlabel("Epoch"); ax.set_ylabel("MSE (scaled)")
ax.legend()
plt.tight_layout()
plt.savefig("training_curve.png", dpi=120)
plt.show()

In [ ]:
def compute_regression_metrics(y_true_scaled, y_pred_scaled, scaler):
    """Inverse-transform and compute MAE, RMSE, MAPE, R²."""
    y_true = scaler.inverse_transform(y_true_scaled.reshape(-1, 1)).flatten()
    y_pred = scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()

    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    r2   = r2_score(y_true, y_pred)
    return {"MAE": mae, "RMSE": rmse, "MAPE": mape, "R2": r2}, y_true, y_pred


# ── Evaluate on test set ───────────────────────────────────────────────────────
preds_scaled, actuals_scaled = predict_all(lstm_model, test_loader)
metrics, y_true_orig, y_pred_orig = compute_regression_metrics(
    actuals_scaled, preds_scaled, scaler
)

print("\n── LSTM Test Metrics ──")
for k, v in metrics.items():
    print(f"  {k:<6}: {v:.4f}")

print("""
Metric Rationale:
  MAPE (Mean Absolute Percentage Error) is the most meaningful metric for a
  trading application because it is scale-invariant across different price levels
  (₹400 WIPRO vs ₹4000 TCS) and directly interpretable as a percentage deviation.

  Deployment bar: A model is worth deploying only if MAPE < 1.5% consistently.
  Below this, the price forecast is more informative than a naive carry-forward
  baseline. Above it, transaction costs eat any potential edge.
""")

In [ ]:
# ── Prediction vs Actual plot ──────────────────────────────────────────────────
test_dates = test_df["date"].iloc[WINDOW_SIZE:].values

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(test_dates, y_true_orig, label="Actual Close",    color="steelblue",  linewidth=1.5)
ax.plot(test_dates, y_pred_orig, label="LSTM Predicted",  color="crimson",    linewidth=1.2, linestyle="--")
ax.set_title(f"{STOCK_TICKER} — LSTM Next-Day Close Prediction (Test Period)", fontsize=13)
ax.set_xlabel("Date"); ax.set_ylabel("Close Price (₹)")
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.tight_layout()
plt.savefig("lstm_predictions.png", dpi=120)
plt.show()

---
## Sub-step 4 🟡 — Churn Prediction *(placeholder — chat_logs.csv required)*

> **Hypothesis to test:** Does the sequential nature of chat interactions add value over aggregated tabular features?  
> Approach: Build (a) an LSTM on raw chat sequences and (b) a Gradient Boosting model on engineered aggregate features (session count, avg response time, sentiment trend, etc.). Compare on ROC-AUC given likely class imbalance in churn datasets.  
> If AUC(GBM) ≥ AUC(LSTM) within 1%, prefer GBM — lower complexity and interpretability win in production.

---
## Sub-step 5 🟡 — Churn Risk Ranking & Cost-Model Threshold

> This sub-step depends on Sub-step 4 model output. The cost-model framework is defined here.

**Cost Model:**
- `C_FP` = cost of contacting a customer who would NOT churn (e.g., discount voucher cost = ₹200)
- `C_FN` = cost of missing a churner (LTV lost = ₹2,000)
- **Optimal threshold** = point where `C_FP × (1 - precision) = C_FN × recall` i.e. where `C_FP / C_FN = precision / (1 - precision)` → threshold ≈ 0.09 for this cost ratio
- At this threshold, outreach is cost-effective: expected savings per contacted customer > ₹0.

```python
# Once churn model is available:
# churn_probs = model.predict_proba(X_test)[:, 1]
# THRESHOLD = 0.09   # derived from C_FP/C_FN ratio
# at_risk = df_test[churn_probs >= THRESHOLD][["customer_id", "churn_prob"]]
# at_risk = at_risk.sort_values("churn_prob", ascending=False)
# print(f"Customers to contact this month: {len(at_risk)}")
```

---
## Sub-step 6 🔴 (Hard) — Autoregressive Baseline vs LSTM

In [ ]:
def autoregressive_predict(prices_scaled: np.ndarray, window: int) -> np.ndarray:
    """Predict next-day price as weighted average of last `window` days.

    Weights decay linearly so more recent days contribute more,
    mimicking an exponentially-weighted moving average.
    """
    weights = np.arange(1, window + 1, dtype=np.float32)
    weights /= weights.sum()  # normalise

    preds = []
    for i in range(len(prices_scaled) - window):
        window_slice = prices_scaled[i : i + window]
        preds.append(np.dot(window_slice, weights))
    return np.array(preds, dtype=np.float32)


# Build AR baseline on same test window
ar_preds_scaled  = autoregressive_predict(test_prices, WINDOW_SIZE)
ar_actuals_scaled = test_prices[WINDOW_SIZE:]

ar_metrics, ar_true, ar_pred = compute_regression_metrics(
    ar_actuals_scaled, ar_preds_scaled, scaler
)

print("── Autoregressive Baseline ──")
for k, v in ar_metrics.items():
    print(f"  {k:<6}: {v:.4f}")

print("\n── LSTM ──")
for k, v in metrics.items():
    print(f"  {k:<6}: {v:.4f}")

In [ ]:
# ── Side-by-side comparison chart ──────────────────────────────────────────────
labels  = list(metrics.keys())
lstm_vals = [metrics[k] for k in labels]
ar_vals   = [ar_metrics[k] for k in labels]

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x - width/2, lstm_vals, width, label="LSTM",                color="steelblue")
ax.bar(x + width/2, ar_vals,   width, label="AR Baseline",         color="darkorange")
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_title("LSTM vs Autoregressive Baseline — Test Set Metrics")
ax.legend()
plt.tight_layout()
plt.savefig("model_comparison.png", dpi=120)
plt.show()

# Diagnosis
if ar_metrics["MAPE"] <= metrics["MAPE"]:
    print("""
[Diagnosis — AR wins or ties]
The AR baseline performs comparably to the LSTM. This likely means:
  1. The close-price series is near a random walk over this period.
  2. 30-day windows do not contain exploitable non-linear structure.
  3. The LSTM is overfitting to noise rather than learning signal.
Recommended next step: add technical indicators (RSI, Bollinger Bands)
as additional input features before concluding LSTMs are ineffective.
""")
else:
    print("""
[Diagnosis — LSTM wins]
The LSTM outperforms the AR baseline, indicating it has learned non-linear
temporal patterns that a weighted moving average cannot capture — likely
momentum reversals or regime changes that are visible only over 20–30 day
windows.
""")

---
## AI Usage Log

As required by the AI Usage Policy, every AI-assisted step is documented here.

| Sub-step | Prompt used | What I changed & why |
|---|---|---|
| 1 | *"Given a stock price DataFrame with columns date, open, high, low, close, volume, returns_pct for tickers RELIANCE/INFOSYS/TCS/HDFC/WIPRO over 2022-2024, write a function to build sliding-window sequences for LSTM input. Explain the correct train/val/test split strategy for time-series."* | AI used a 70/30 split; I changed to 80/10/10 and added a validation set for early stopping. AI omitted scaler fitting only on train data — fixed to prevent leakage. |
| 3 | *"Write a PyTorch LSTM class for univariate time-series regression. Include training loop with early stopping."* | AI used a 3-layer LSTM without dropout — reduced to 2 layers + 0.2 dropout to prevent overfitting on ~600 sequences. Added gradient clipping (max_norm=1.0) which AI omitted. |
| 6 | *"Compare an LSTM to a weighted autoregressive baseline for next-day stock price prediction. Include a diagnostic for why each model might win."* | AI's baseline used equal weights — changed to linearly increasing weights (recency bias) for a fairer and more realistic AR model. |

**Critique:** AI outputs were structurally sound but consistently skipped data-leakage safeguards (fitting the scaler on full data, using random splits). Engineering Quality decisions required manual correction in every sub-step.